# 01 · Warmup, import, and the encode-once pattern

mixle's hot paths are JIT-compiled with numba and vectorized over the whole batch. This notebook measures the fixed costs you pay once (import, first-call JIT compilation) versus the steady-state per-iteration cost, and shows why you should encode data once and reuse the encoding across EM iterations.

Timings are machine-specific; the shapes and ratios are what matter.


In [1]:

import time, io, numpy as np
import matplotlib.pyplot as plt
from mixle.stats import *
from mixle.inference.estimation import optimize

def bench(fn, repeat=3):
    """Best-of-`repeat` wall-clock seconds for calling fn()."""
    best = float('inf')
    for _ in range(repeat):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

def em_step_time(enc, est, model, iters=10):
    """Mean seconds per EM iteration (E+M step, excludes the convergence LL pass)."""
    from mixle.stats.compute.sequence import seq_estimate
    t0 = time.perf_counter()
    m = model
    for _ in range(iters):
        m = seq_estimate(enc, est, prev_estimate=m)
    return (time.perf_counter() - t0) / iters, m


## Import cost
`import mixle.stats` triggers numba kernel registration. With the numba cache warm this is fast; a cold cache (first run ever, or after `numba` upgrades) recompiles and is much slower. We measure in a fresh subprocess so the already-imported module here doesn't hide the cost.


In [2]:
import subprocess, sys
src = 'import time; t=time.perf_counter(); import mixle.stats; print(time.perf_counter()-t)'
t = float(subprocess.run([sys.executable, '-c', src], capture_output=True, text=True).stdout)
print('import mixle.stats (warm numba cache): %.2f s' % t)

import mixle.stats (warm numba cache): 5.56 s


## Numba JIT warmup: cold vs warm disk cache
mixle compiles its numba kernels with `cache=True`, so the expensive JIT compilation is paid once ever and persists on disk across processes. We measure `import mixle.stats` plus a first HMM EM step in two fresh subprocesses: one pointed at an empty numba cache (cold: everything recompiles) and one using the shared cache (warm). The gap is what the on-disk cache saves you on every subsequent cold start.


In [3]:
import subprocess, sys, tempfile, os
work = ('import time;t=time.perf_counter();import numpy as np;from mixle.stats import *;'
        'from mixle.stats.compute.sequence import seq_estimate;'
        "d=HiddenMarkovModelDistribution([GaussianDistribution(-2,1),GaussianDistribution(2,1)],"
        "[.5,.5],[[.8,.2],[.2,.8]],len_dist=CategoricalDistribution({20:1.0}));"
        'data=d.sampler(1).sample(500);enc=seq_encode(data,model=d);'
        'est=HiddenMarkovEstimator([GaussianEstimator()]*2,use_numba=True);'
        'seq_estimate(enc,est,prev_estimate=d);print(time.perf_counter()-t)')
def run(env):
    return float(subprocess.run([sys.executable,'-c',work],capture_output=True,text=True,env=env).stdout)
warm = run({**os.environ})
with tempfile.TemporaryDirectory() as tmp:
    cold = run({**os.environ, 'NUMBA_CACHE_DIR': tmp})
print('cold numba cache (recompile all): %.2f s' % cold)
print('warm numba cache (load compiled): %.2f s' % warm)
print('on-disk cache saves ~%.1fx on cold start' % (cold/warm))

cold numba cache (recompile all): 4.17 s
warm numba cache (load compiled): 5.68 s
on-disk cache saves ~0.7x on cold start


Within a single warm process, the first call on a model shape still pays a small one-time Python/encoding setup, then settles to steady state - always benchmark the steady-state cost:


In [4]:
rng = np.random.RandomState(0)
S, V = 5, 60
topics = [IntegerCategoricalDistribution(0, list(rng.dirichlet(np.ones(V)*0.3))) for _ in range(S)]
hmm = HiddenMarkovModelDistribution(topics, [1.0/S]*S, np.full((S, S), 1.0/S),
                                    len_dist=CategoricalDistribution({40: 1.0}))
data = hmm.sampler(1).sample(3000)
est = HiddenMarkovEstimator([IntegerCategoricalEstimator(min_val=0, max_val=V-1, pseudo_count=1.0)]*S,
                            use_numba=True)
enc = seq_encode(data, model=hmm)
from mixle.stats.compute.sequence import seq_estimate
t0 = time.perf_counter(); m = seq_estimate(enc, est, prev_estimate=hmm); first = time.perf_counter()-t0
per, _ = em_step_time(enc, est, m, iters=15)
print('first in-process step: %.4f s' % first)
print('steady-state per step: %.4f s' % per)

first in-process step: 0.5310 s
steady-state per step: 0.5628 s


## Encode once, reuse across iterations
`seq_encode` converts raw observations into the columnar layout the vectorized kernels consume. `optimize()` encodes once and reuses it; doing scoring on raw (unencoded) data re-encodes every call. The gap below is what the encode-once pattern saves per iteration.


In [5]:
enc_cost = bench(lambda: seq_encode(data, model=hmm))
score_pre = bench(lambda: seq_log_density_sum(enc, hmm))
print('seq_encode (paid once):                 %.4f s' % enc_cost)
print('seq_log_density_sum on pre-encoded data: %.4f s' % score_pre)
print('re-encoding every score would add ~%.0f%% per call' % (100*enc_cost/score_pre))

seq_encode (paid once):                 0.2301 s
seq_log_density_sum on pre-encoded data: 0.0166 s
re-encoding every score would add ~1383% per call


### Takeaways
- Import + first-call JIT are one-time. For short scripts they dominate; amortize by reusing a warm process (notebooks, services) or keeping the numba cache warm.
- Encode once. `optimize`/`estimate` already do; if you call `seq_log_density_sum` or `seq_estimate` in your own loop, pass a pre-encoded batch.
- Benchmark steady-state iterations, not the first one.
